# 因果多头自注意力（Causal MHA）

源码导航：[`core/attention/mha.py`](../../../core/attention/mha.py)。

## 1. 理论推导

### Scaled Dot-Product Attention

给定 Query $Q \in \mathbb{R}^{T \times d_k}$、Key $K \in \mathbb{R}^{T \times d_k}$、Value $V \in \mathbb{R}^{T \times d_v}$，缩放点积注意力定义为：

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right) V$$

其中 $\sqrt{d_k}$ 的缩放防止点积随维度增大而数值过大（方差为 $d_k$ 的分布缩放到方差为 1），$M$ 为**因果掩码**：

$$M_{ij} = \begin{cases} 0 & i \geq j \\ -\infty & i < j \end{cases}$$

经 softmax 后，$i < j$ 处的权重趋近于 0，实现"当前位置只能看到自身及历史 token"的因果约束。

### 多头投影

MHA 将 $d$ 维空间拆分为 $H$ 个 $d_h = d/H$ 维子空间，各头独立计算注意力后拼接：

$$\text{MultiHead}(x) = \text{Concat}(\text{head}_1, \ldots, \text{head}_H) \cdot W_O$$

$$\text{head}_h = \text{Attention}(xW_Q^h,\ xW_K^h,\ xW_V^h)$$

**`c_attn` 优化**：GPT-2 将 $H$ 组 $W_Q, W_K, W_V$ 合并为一个 $\mathbb{R}^{d \times 3d}$ 矩阵一次性计算，再按 $d$ 维度拆成三份，减少三次矩阵乘法的 kernel 调度开销：

$$[Q; K; V] = x \cdot W_\text{QKV}, \quad W_\text{QKV} \in \mathbb{R}^{d \times 3d}$$

### 残差路径的初始化缩放

GPT-2 论文指出，对深层网络的残差路径输出投影 `c_proj`，初始化标准差应按层数缩放：

$$\sigma_\text{c\_proj} = \frac{\sigma_0}{\sqrt{2N}}$$

其中 $N$ 为 Transformer 块总数。这使初始化时各层残差贡献的方差之和为 $\sigma_0^2$，防止深层网络激活值漂移。

In [ ]:
from __future__ import annotations

import sys
import math
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['font.family'] = 'DejaVu Sans'

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.attention.mha import CausalSelfAttention

因果注意力权重矩阵 (Softmax 后):
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.4716, 0.5284, 0.0000, 0.0000],
        [0.2834, 0.4954, 0.2212, 0.0000],
        [0.0600, 0.1093, 0.7151, 0.1156]])


### 2. 因果注意力权重热图

以一个序列长度 $T=8$ 的单头 eager attention 为例，可视化 softmax 后的注意力权重矩阵，验证其严格下三角结构（上三角 = 0）。

In [ ]:
torch.manual_seed(0)
T, d_k = 8, 16
q = torch.randn(T, d_k)
k = torch.randn(T, d_k)

# 手工实现 eager attention，直观验证因果掩码
att = (q @ k.T) / math.sqrt(d_k)                               # (T, T) 原始分数
causal_mask = torch.tril(torch.ones(T, T, dtype=torch.bool))   # 下三角 True
att = att.masked_fill(~causal_mask, float('-inf'))              # 上三角置 -inf
att_softmax = F.softmax(att, dim=-1)                            # softmax 后上三角 = 0

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(att_softmax.detach().numpy(), vmin=0, vmax=1, cmap='Blues')
ax.set_title('Causal Attention Weights (T=8)', fontsize=11)
ax.set_xlabel('Key Position'); ax.set_ylabel('Query Position')
plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

# 验证上三角严格为 0
upper = att_softmax[~causal_mask]
print(f"上三角元素最大值（应为 0）: {upper.max().item():.2e}")

### 3. Sanity Check：`eager` 与 `sdpa` 输出等价性

`sdpa` 路径使用 `F.scaled_dot_product_attention(is_causal=True)` 调用 FlashAttention 内核；`eager` 路径是手写的等价实现。两者对同一输入的输出应数值上等价（误差在浮点精度范围内）。

In [ ]:
torch.manual_seed(1)
B, T, n_embd, n_head = 2, 12, 64, 4

attn_eager = CausalSelfAttention(
    n_embd=n_embd, n_head=n_head, block_size=64,
    dropout=0.0, bias=True, attn_impl='eager',
)
attn_sdpa = CausalSelfAttention(
    n_embd=n_embd, n_head=n_head, block_size=64,
    dropout=0.0, bias=True, attn_impl='sdpa',
)
# 共享权重，确保比较在同一参数下进行
attn_sdpa.load_state_dict(attn_eager.state_dict())

x = torch.randn(B, T, n_embd)
with torch.no_grad():
    y_eager = attn_eager(x)
    y_sdpa  = attn_sdpa(x)

max_diff = (y_eager - y_sdpa).abs().max().item()
print(f"eager vs sdpa 最大绝对差: {max_diff:.2e}  (应 < 1e-5)")
assert max_diff < 1e-4, f"差值过大: {max_diff}"

# c_attn 权重验证：shape = (n_embd, 3*n_embd)
print(f"c_attn weight shape: {tuple(attn_eager.c_attn.weight.shape)}")
print(f"head_dim = n_embd / n_head = {n_embd} / {n_head} = {n_embd // n_head}")

### 4. 源码精讲

**`c_attn` 融合投影与多头 reshape**（`CausalSelfAttention.forward`）：

```python
# c_attn: Linear(n_embd, 3*n_embd) 一次性算出 Q/K/V
qkv = self.c_attn(x)                                             # (B, T, 3*C)
q, k, v = qkv.split(self.n_embd, dim=2)                         # 各 (B, T, C)

# reshape 成 (B, H, T, d_h) 以便按头并行计算
q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)    # (B, H, T, d_h)
k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
```

**sdpa 路径**（默认，自动选择 FlashAttention 内核）：

```python
y = F.scaled_dot_product_attention(
    q, k, v,
    attn_mask=None,
    dropout_p=self.dropout if self.training else 0.0,
    is_causal=True,          # PyTorch 内部构造因果 mask，避免显式分配 O(T²) 掩码矩阵
)
```

**eager 路径**（教学 / sanity check 用）：

```python
att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)      # (B, H, T, T) 原始分数
# 注册为 buffer 的下三角掩码 _causal_mask：persistent=False 不写入 state_dict
att = att.masked_fill(~self._causal_mask[:, :, :T, :T], float('-inf'))
att = F.softmax(att, dim=-1)
att = self.attn_dropout(att)
y = att @ v                                                       # (B, H, T, d_h)
```

**输出 reshape 与 c_proj**：

```python
# 合并多头：(B, H, T, d_h) → (B, T, C)
y = y.transpose(1, 2).contiguous().view(B, T, C)
y = self.c_proj(y)    # 输出投影，init std = init_std / sqrt(2*N)（残差路径缩放）
y = self.resid_dropout(y)
```

---

## 延伸阅读与参考资料

### 核心论文
- **Attention Is All You Need**: Vaswani et al., 2017. [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
- **FlashAttention**: Dao et al., 2022. [arXiv:2205.14135](https://arxiv.org/abs/2205.14135)
- **FlashAttention-2**: Dao, 2023. [arXiv:2307.08691](https://arxiv.org/abs/2307.08691)

### 参考实现
- **PyTorch SDPA 文档**: [scaled_dot_product_attention](https://pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)
- **nanoGPT `CausalSelfAttention`**: [GitHub](https://github.com/karpathy/nanoGPT/blob/master/model.py)